In [ ]:
import os, json, math, shutil, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

In [ ]:
import os
import json
import math
import shutil
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import networkx as nx
except ImportError as e:
    raise ImportError("This script requires networkx. Install with `pip install networkx`.") from e

CONFIG = {
    "input_useful_post_file": "stage2_regime_results_useful_with_post_analysis.csv",
    "input_presentation_file": "stage2_regime_results_presentation_subset.csv",
    "issuer_id_cols": ["company_symbol_1", "company_symbol_2"],
    "sector_cols": ["sector_1", "sector_2"],
    "pair_id_cols": ["cusip_1", "cusip_2"],
    "subset_modes": ["useful_all", "high_quality", "presentation"],
    "min_pairs_per_issuer_edge_useful_all": 1,
    "min_pairs_per_issuer_edge_high_quality": 1,
    "min_pairs_per_issuer_edge_presentation": 1,
    "edge_weight_components": {
        "trace_margin_95": 1.00,
        "spread_change_corr_shortlist": 1.00,
        "min_regime_occupancy": 0.80,
        "variance_ratio_capped": 1.10,
        "phi_gap": 1.10,
        "inverse_maturity_diff": 0.80,
        "inverse_half_life_gap_to_target": 0.70,
    },
    "half_life_target_days": 3.0,
    "variance_ratio_cap": 8.0,
    "maturity_diff_cap": 2.0,
    "top_n_edges_per_subset": 250,
    "top_n_nodes_per_subset": 100,
    "community_detection": True,
    "spring_layout_seed": 42,
    "output_dir": "stage3_network_mapping_outputs",
    "zip_basename": "stage3_network_mapping_outputs",
    "zip_outputs": True,
    "auto_download_zip_if_colab": True,
}

# ---------- utilities ----------
def safe_numeric_rank(s: pd.Series, ascending: bool = True) -> pd.Series:
    if s.isna().all():
        return pd.Series(np.full(len(s), 0.5), index=s.index)
    return s.rank(pct=True, ascending=ascending, method="average").fillna(0.5)

def normalize_inverse(series: pd.Series, cap: float = None) -> pd.Series:
    x = series.astype(float).copy()
    if cap is not None:
        x = np.minimum(x, cap)
    return 1.0 / (1.0 + x)

def normalize_closeness_to_target(series: pd.Series, target: float) -> pd.Series:
    x = (series.astype(float) - target).abs()
    return 1.0 / (1.0 + x)

def try_download_in_colab(path: Path):
    if not CONFIG.get("auto_download_zip_if_colab", False):
        return
    try:
        import google.colab  # type: ignore
        from google.colab import files  # type: ignore
        files.download(str(path))
    except Exception:
        pass

def make_zip(output_dir: Path, zip_path: Path):
    if zip_path.exists():
        zip_path.unlink()
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for item in output_dir.rglob("*"):
            if item == zip_path:
                continue
            if item.is_file():
                zf.write(item, item.relative_to(output_dir.parent))

def compute_edge_score(df: pd.DataFrame) -> pd.DataFrame:
    work = df.copy()

    work["variance_ratio_capped"] = np.minimum(work["variance_ratio"].astype(float), CONFIG["variance_ratio_cap"])
    work["inverse_maturity_diff"] = normalize_inverse(work["avg_remaining_maturity_diff"], CONFIG["maturity_diff_cap"])
    work["inverse_half_life_gap_to_target"] = normalize_closeness_to_target(work["half_life_days_stage1"], CONFIG["half_life_target_days"])

    comp_weights = CONFIG["edge_weight_components"]
    score = pd.Series(0.0, index=work.index)
    total_w = 0.0

    for col, w in comp_weights.items():
        if col not in work.columns:
            continue
        if col in ["inverse_maturity_diff", "inverse_half_life_gap_to_target"]:
            comp = work[col].astype(float).fillna(work[col].median())
        else:
            comp = safe_numeric_rank(work[col].astype(float), ascending=True)
        score += w * comp
        total_w += w

    if total_w == 0:
        work["network_pair_score"] = 0.0
    else:
        work["network_pair_score"] = score / total_w

    return work

def prepare_subset(df_useful: pd.DataFrame, df_presentation: pd.DataFrame, subset_mode: str) -> pd.DataFrame:
    if subset_mode == "useful_all":
        df = df_useful.copy()
    elif subset_mode == "high_quality":
        df = df_useful[df_useful["high_quality_useful"] == True].copy()
    elif subset_mode == "presentation":
        df = df_presentation.copy()
    else:
        raise ValueError(f"Unknown subset_mode: {subset_mode}")

    df = compute_edge_score(df)

    issuer1, issuer2 = CONFIG["issuer_id_cols"]
    sector1, sector2 = CONFIG["sector_cols"]
    cusip1, cusip2 = CONFIG["pair_id_cols"]

    # canonical issuer ordering so pair is undirected
    ordered_issuer = np.where(df[issuer1] <= df[issuer2], df[issuer1], df[issuer2])
    ordered_issuer_other = np.where(df[issuer1] <= df[issuer2], df[issuer2], df[issuer1])
    ordered_sector = np.where(df[issuer1] <= df[issuer2], df[sector1], df[sector2])
    ordered_sector_other = np.where(df[issuer1] <= df[issuer2], df[sector2], df[sector1])

    df["issuer_a"] = ordered_issuer
    df["issuer_b"] = ordered_issuer_other
    df["sector_a"] = ordered_sector
    df["sector_b"] = ordered_sector_other
    df["issuer_pair_key"] = df["issuer_a"].astype(str) + "__" + df["issuer_b"].astype(str)
    df["bond_pair_key"] = df[cusip1].astype(str) + "__" + df[cusip2].astype(str)

    return df

def aggregate_issuer_edges(df: pd.DataFrame, subset_mode: str) -> pd.DataFrame:
    gcols = ["issuer_a", "issuer_b", "sector_a", "sector_b"]
    grp = df.groupby(gcols, dropna=False)

    edges = grp.agg(
        n_bond_pairs=("bond_pair_key", "nunique"),
        mean_network_pair_score=("network_pair_score", "mean"),
        median_network_pair_score=("network_pair_score", "median"),
        max_network_pair_score=("network_pair_score", "max"),
        median_trace_margin_95=("trace_margin_95", "median"),
        median_spread_change_corr=("spread_change_corr_shortlist", "median"),
        median_maturity_diff=("avg_remaining_maturity_diff", "median"),
        median_half_life_stage1=("half_life_days_stage1", "median"),
        median_variance_ratio=("variance_ratio", "median"),
        median_phi_gap=("phi_gap", "median"),
        median_min_regime_occupancy=("min_regime_occupancy", "median"),
        share_high_quality=("high_quality_useful", "mean"),
        share_both_variance_and_reversion_shift=("regime_distinction_type", lambda x: (x == "both_variance_and_reversion_shift").mean()),
        share_both_regimes_mean_reverting=("reversion_pattern_type", lambda x: (x == "both_regimes_mean_reverting").mean()),
    ).reset_index()

    edges["edge_weight"] = (
        0.70 * edges["median_network_pair_score"].astype(float)
        + 0.30 * safe_numeric_rank(edges["n_bond_pairs"].astype(float), ascending=True)
    )
    edges["subset_mode"] = subset_mode
    edges["sector_combo"] = np.where(
        edges["sector_a"] <= edges["sector_b"],
        edges["sector_a"].astype(str) + " | " + edges["sector_b"].astype(str),
        edges["sector_b"].astype(str) + " | " + edges["sector_a"].astype(str),
    )

    min_pairs = CONFIG.get(f"min_pairs_per_issuer_edge_{subset_mode}", 1)
    edges = edges[edges["n_bond_pairs"] >= min_pairs].copy()
    edges = edges.sort_values(["edge_weight", "n_bond_pairs"], ascending=[False, False]).reset_index(drop=True)
    edges["edge_rank"] = np.arange(1, len(edges) + 1)
    return edges

def aggregate_sector_edges(edges: pd.DataFrame, subset_mode: str) -> pd.DataFrame:
    grp = edges.groupby("sector_combo", dropna=False)
    out = grp.agg(
        n_issuer_edges=("edge_rank", "count"),
        median_edge_weight=("edge_weight", "median"),
        mean_edge_weight=("edge_weight", "mean"),
        total_bond_pairs=("n_bond_pairs", "sum"),
        median_variance_ratio=("median_variance_ratio", "median"),
        median_phi_gap=("median_phi_gap", "median"),
        median_min_regime_occupancy=("median_min_regime_occupancy", "median"),
    ).reset_index()
    out["subset_mode"] = subset_mode
    out = out.sort_values(["median_edge_weight", "n_issuer_edges"], ascending=[False, False]).reset_index(drop=True)
    out["sector_combo_rank"] = np.arange(1, len(out) + 1)
    return out

def build_graph(edges: pd.DataFrame):
    G = nx.Graph()
    for _, row in edges.iterrows():
        a, b = row["issuer_a"], row["issuer_b"]
        G.add_node(a, sector=row["sector_a"])
        G.add_node(b, sector=row["sector_b"])
        G.add_edge(
            a, b,
            weight=float(row["edge_weight"]),
            n_bond_pairs=int(row["n_bond_pairs"]),
            median_trace_margin_95=float(row["median_trace_margin_95"]),
            median_variance_ratio=float(row["median_variance_ratio"]),
            median_phi_gap=float(row["median_phi_gap"]),
        )
    return G

def node_summary_from_graph(G, edges: pd.DataFrame, subset_mode: str) -> pd.DataFrame:
    if G.number_of_nodes() == 0:
        return pd.DataFrame(columns=["issuer","sector","degree","weighted_degree","betweenness","eigenvector","clustering","subset_mode"])
    degree = dict(G.degree())
    weighted_degree = dict(G.degree(weight="weight"))
    betweenness = nx.betweenness_centrality(G, weight="weight", normalized=True)
    try:
        eigenvector = nx.eigenvector_centrality_numpy(G, weight="weight")
    except Exception:
        eigenvector = {n: np.nan for n in G.nodes()}
    clustering = nx.clustering(G, weight="weight")

    node_df = pd.DataFrame({
        "issuer": list(G.nodes()),
        "sector": [G.nodes[n].get("sector") for n in G.nodes()],
        "degree": [degree.get(n, 0) for n in G.nodes()],
        "weighted_degree": [weighted_degree.get(n, 0.0) for n in G.nodes()],
        "betweenness": [betweenness.get(n, np.nan) for n in G.nodes()],
        "eigenvector": [eigenvector.get(n, np.nan) for n in G.nodes()],
        "clustering": [clustering.get(n, np.nan) for n in G.nodes()],
        "subset_mode": subset_mode,
    })

    if CONFIG.get("community_detection", True):
        try:
            comms = nx.algorithms.community.greedy_modularity_communities(G, weight="weight")
            membership = {}
            for i, comm in enumerate(comms, start=1):
                for node in comm:
                    membership[node] = i
            node_df["community_id"] = node_df["issuer"].map(membership)
            node_df["n_communities"] = len(comms)
        except Exception:
            node_df["community_id"] = np.nan
            node_df["n_communities"] = np.nan
    else:
        node_df["community_id"] = np.nan
        node_df["n_communities"] = np.nan

    node_df = node_df.sort_values(["weighted_degree", "degree", "betweenness"], ascending=[False, False, False]).reset_index(drop=True)
    node_df["node_rank"] = np.arange(1, len(node_df) + 1)
    return node_df

def plot_graph(G, node_df: pd.DataFrame, outpath: Path, title: str):
    if G.number_of_nodes() == 0:
        return
    plt.figure(figsize=(13, 10))
    pos = nx.spring_layout(G, seed=CONFIG["spring_layout_seed"], weight="weight", k=None)
    nodes = list(G.nodes())
    node_sizes = []
    for n in nodes:
        wd = node_df.loc[node_df["issuer"] == n, "weighted_degree"]
        size = 120 + (wd.iloc[0] * 1800 if not wd.empty else 200)
        node_sizes.append(size)
    edge_widths = [0.8 + 4.0 * G[u][v]["weight"] for u, v in G.edges()]
    nx.draw_networkx_edges(G, pos, alpha=0.28, width=edge_widths)
    nx.draw_networkx_nodes(G, pos, node_size=node_sizes, alpha=0.88)
    labels = {row["issuer"]: row["issuer"] for _, row in node_df.head(25).iterrows()}
    nx.draw_networkx_labels(G, pos, labels=labels, font_size=9)
    plt.title(title)
    plt.axis("off")
    plt.tight_layout()
    plt.savefig(outpath, dpi=220, bbox_inches="tight")
    plt.close()

def save_subset_outputs(df_subset, edges, sector_edges, node_df, subset_dir: Path, subset_mode: str):
    subset_dir.mkdir(parents=True, exist_ok=True)
    df_subset.to_csv(subset_dir / f"network_input_pairs_{subset_mode}.csv", index=False)
    edges.to_csv(subset_dir / f"issuer_network_edges_{subset_mode}.csv", index=False)
    sector_edges.to_csv(subset_dir / f"sector_network_edges_{subset_mode}.csv", index=False)
    node_df.to_csv(subset_dir / f"issuer_network_nodes_{subset_mode}.csv", index=False)

    top_n_edges = CONFIG["top_n_edges_per_subset"]
    top_n_nodes = CONFIG["top_n_nodes_per_subset"]
    edges.head(top_n_edges).to_csv(subset_dir / f"top_issuer_network_edges_{subset_mode}.csv", index=False)
    node_df.head(top_n_nodes).to_csv(subset_dir / f"top_issuer_network_nodes_{subset_mode}.csv", index=False)

    G = build_graph(edges)
    plot_graph(G, node_df, subset_dir / f"issuer_network_graph_{subset_mode}.png", f"Issuer network: {subset_mode}")

def main():
    outdir = Path(CONFIG["output_dir"])
    outdir.mkdir(parents=True, exist_ok=True)

    useful = pd.read_csv(CONFIG["input_useful_post_file"])
    presentation = pd.read_csv(CONFIG["input_presentation_file"])

    all_reports = []
    for subset_mode in CONFIG["subset_modes"]:
        df_subset = prepare_subset(useful, presentation, subset_mode)
        edges = aggregate_issuer_edges(df_subset, subset_mode)
        sector_edges = aggregate_sector_edges(edges, subset_mode)
        G = build_graph(edges)
        node_df = node_summary_from_graph(G, edges, subset_mode)

        subset_dir = outdir / subset_mode
        save_subset_outputs(df_subset, edges, sector_edges, node_df, subset_dir, subset_mode)

        report_row = {
            "subset_mode": subset_mode,
            "n_pair_rows_in_subset": int(len(df_subset)),
            "n_issuer_edges": int(len(edges)),
            "n_sector_combos": int(sector_edges["sector_combo"].nunique()) if len(sector_edges) else 0,
            "n_nodes": int(len(node_df)),
            "graph_density": float(nx.density(G)) if G.number_of_nodes() > 1 else np.nan,
            "n_connected_components": int(nx.number_connected_components(G)) if G.number_of_nodes() > 0 else 0,
            "largest_component_nodes": int(max((len(c) for c in nx.connected_components(G)), default=0)) if G.number_of_nodes() > 0 else 0,
            "median_edge_weight": float(edges["edge_weight"].median()) if len(edges) else np.nan,
            "median_n_bond_pairs_per_edge": float(edges["n_bond_pairs"].median()) if len(edges) else np.nan,
            "median_weighted_degree": float(node_df["weighted_degree"].median()) if len(node_df) else np.nan,
            "top_node_issuer": node_df.iloc[0]["issuer"] if len(node_df) else None,
            "top_node_weighted_degree": float(node_df.iloc[0]["weighted_degree"]) if len(node_df) else np.nan,
            "top_edge_issuer_a": edges.iloc[0]["issuer_a"] if len(edges) else None,
            "top_edge_issuer_b": edges.iloc[0]["issuer_b"] if len(edges) else None,
            "top_edge_weight": float(edges.iloc[0]["edge_weight"]) if len(edges) else np.nan,
        }
        all_reports.append(report_row)

    report_df = pd.DataFrame(all_reports)
    report_df.to_csv(outdir / "stage3_network_mapping_report.csv", index=False)

    # combined top tables for convenience
    combined_nodes = []
    combined_edges = []
    combined_sector = []
    for subset_mode in CONFIG["subset_modes"]:
        subset_dir = outdir / subset_mode
        combined_nodes.append(pd.read_csv(subset_dir / f"issuer_network_nodes_{subset_mode}.csv"))
        combined_edges.append(pd.read_csv(subset_dir / f"issuer_network_edges_{subset_mode}.csv"))
        combined_sector.append(pd.read_csv(subset_dir / f"sector_network_edges_{subset_mode}.csv"))
    pd.concat(combined_nodes, ignore_index=True).to_csv(outdir / "stage3_network_nodes_all_subsets.csv", index=False)
    pd.concat(combined_edges, ignore_index=True).to_csv(outdir / "stage3_network_edges_all_subsets.csv", index=False)
    pd.concat(combined_sector, ignore_index=True).to_csv(outdir / "stage3_sector_network_edges_all_subsets.csv", index=False)

    readme = """Stage 3 network mapping outputs
================================

This stage converts the post-regime useful pairs into issuer-level and sector-level networks.

Subsets produced:
- useful_all: all useful regime pairs with post-analysis
- high_quality: useful pairs flagged as high_quality_useful
- presentation: thesis-ready presentation subset

For each subset the script outputs:
- network_input_pairs_<subset>.csv
- issuer_network_edges_<subset>.csv
- issuer_network_nodes_<subset>.csv
- sector_network_edges_<subset>.csv
- top_issuer_network_edges_<subset>.csv
- top_issuer_network_nodes_<subset>.csv
- issuer_network_graph_<subset>.png

Key ideas:
- Nodes are issuers (company symbols)
- Edges aggregate many bond-pair results between two issuers
- Edge weights combine Stage 1/Stage 2 strength and economic comparability
- Node summaries report degree, weighted degree, betweenness, eigenvector centrality, clustering, and community ids when available
"""
    (outdir / "README_stage3_network_mapping.txt").write_text(readme)
    (outdir / "stage3_network_mapping_config.json").write_text(json.dumps(CONFIG, indent=2))

    if CONFIG["zip_outputs"]:
        zip_path = outdir / f"{CONFIG['zip_basename']}.zip"
        make_zip(outdir, zip_path)
        try_download_in_colab(zip_path)

    print("Saved network mapping outputs to:", outdir.resolve())
    print(report_df)

if __name__ == "__main__":
    main()

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Saved network mapping outputs to: /content/stage3_network_mapping_outputs
    subset_mode  n_pair_rows_in_subset  n_issuer_edges  n_sector_combos  \
0    useful_all                    442             357               54   
1  high_quality                    252             220               52   
2  presentation                     96              92               38   

   n_nodes  graph_density  n_connected_components  largest_component_nodes  \
0       65       0.171635                       1                       65   
1       63       0.112647                       1                       63   
2       59       0.053770                       1                       59   

   median_edge_weight  median_n_bond_pairs_per_edge  median_weighted_degree  \
0            0.502933                           1.0                5.683781   
1            0.507187                           1.0                3.528200   
2            0.516635                           1.0                1.761771

In [ ]:
main()

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Saved network mapping outputs to: /content/stage3_network_mapping_outputs
    subset_mode  n_pair_rows_in_subset  n_issuer_edges  n_sector_combos  \
0    useful_all                    442             357               54   
1  high_quality                    252             220               52   
2  presentation                     96              92               38   

   n_nodes  graph_density  n_connected_components  largest_component_nodes  \
0       65       0.171635                       1                       65   
1       63       0.112647                       1                       63   
2       59       0.053770                       1                       59   

   median_edge_weight  median_n_bond_pairs_per_edge  median_weighted_degree  \
0            0.502933                           1.0                5.683781   
1            0.507187                           1.0                3.528200   
2            0.516635                           1.0                1.761771